In [ ]:
import sys, os, re, time, warnings, ast
from collections import Counter

import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
from atproto import Client as BskyClient

warnings.filterwarnings("ignore")

In [ ]:
import os, sys

config_path = os.path.join("..", "..", "A. Lectures", "Lecture 1 Social network analysis")
sys.path.insert(0, config_path)
from bsky_config_students import BLUESKY_USERNAME, BLUESKY_APP_PASSWORD

client = BskyClient()
client.login(BLUESKY_USERNAME, BLUESKY_APP_PASSWORD)
print(f"Authenticated as: {BLUESKY_USERNAME}")

In [ ]:
ELECTIONS = {
    "US_2024": {
        "since"        : "2024-07-05T00:00:00Z",
        "until"        : "2024-11-04T23:59:59Z",
        "election_date": "2024-11-05",
        "hashtags": [
            # ── Core election ──────────────────────────────────────────────
            "#Election2024", "#USElection2024", "#ElectionDay", "#Vote2024",
            "#Presidential2024", "#PresidentialElection", "#ElectionNight",
            "#AmericaDecides", "#Decision2024", "#VoterRegistration",
            "#EarlyVoting", "#MailInVoting", "#ElectionIntegrity",
            "#Battleground2024", "#SwingState",

            # ── Trump / Republican ─────────────────────────────────────────
            "#Trump2024", "#TrumpVance", "#VoteTrump", "#Trump",
            "#DonaldTrump", "#MAGA", "#MAGA2024", "#AmericaFirst",
            "#TrumpRally", "#Republicans", "#GOP", "#Republican",
            "#JDVance", "#Vance2024", "#VanceVP",
            "#RNC2024", "#RepublicanConvention",
            "#Project2025", "#TrumpDebate",

            # ── Harris / Democrat ──────────────────────────────────────────
            "#Harris2024", "#KamalaHarris2024", "#KamalaHarris",
            "#HarrisWalz", "#VoteHarris", "#VoteBlue", "#VoteKamala",
            "#Kamala2024", "#Kamala", "#Harris",
            "#TimWalz", "#Walz2024", "#WalzVP",
            "#DNC2024", "#DemConvention", "#DemocraticConvention",
            "#WeAreNotGoingBack", "#WinWithKamala",
            "#Democrats", "#Democrat",

            # ── Biden exit ─────────────────────────────────────────────────
            "#BidenDropsOut", "#BidenOut",
            "#BidenWithdraws",

            # ── Debates & key moments ──────────────────────────────────────
            "#PresidentialDebate", "#Debate2024", "#VPDebate",
            "#TrumpHarrisDebate", "#DebateNight",
        ],
    },
}

ACTIVE_ELECTION = "US_2024"
cfg             = ELECTIONS[ACTIVE_ELECTION]
MIN_TEXT_LENGTH = 30

print(f"Active election : {ACTIVE_ELECTION}")
print(f"Window          : {cfg['since'][:10]}  to  {cfg['until'][:10]}")
print(f"Hashtags        : {len(cfg['hashtags'])}")

In [ ]:
# ── Text cleaning ─────────────────────────────────────────────────────────────
def clean_text(text):
    """Strip URLs and collapse whitespace."""
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


# ── Quality filters ────────────────────────────────────────────────────────────
NEWS_BOT_PATTERNS = [
    "forbes", "guardian", "bbc", "cnn", "reuters", "apnews",
    "skynews", "france24", "euronews", "washingtonpost", "nytimes",
    "independent", "trtworld", "straits_times", "abc", "news-feed",
    "theguardian", "huffpost", "politico", "axios",
]

def is_news_bot(handle):
    return any(p in handle.lower() for p in NEWS_BOT_PATTERNS)

def has_real_text(text):
    """Return True if text has enough non-hashtag/mention content."""
    clean = re.sub(r"#\w+|@\w+", "", text).strip()
    return len(clean) >= MIN_TEXT_LENGTH


# ── Candidate labelling ────────────────────────────────────────────────────────
TRUMP_KW  = ["trump", "maga", "donald", "republican", "gop", "conservative", "vance"]
HARRIS_KW = ["harris", "kamala", "democrat", "democratic", "walz", "liberal", "voteblue"]

def label_candidate(row):
    text    = row["text"].lower()
    score_a = sum(k in text for k in TRUMP_KW)
    score_b = sum(k in text for k in HARRIS_KW)
    if score_a > score_b: return "CandidateA"
    if score_b > score_a: return "CandidateB"
    return "Neutral"


# ── Post search ────────────────────────────────────────────────────────────────
def search_posts(client, query, since=None, until=None):
    """Paginate through all posts matching a hashtag query."""
    collected, cursor = [], None
    while True:
        params = {"q": query, "limit": 100, "sort": "latest"}
        if cursor: params["cursor"] = cursor
        if since:  params["since"]  = since
        if until:  params["until"]  = until
        try:
            resp = client.app.bsky.feed.search_posts(params)
        except Exception as e:
            print(f"  API error: {e}"); break
        if not resp.posts: break
        for post in resp.posts:
            rec  = post.record
            raw  = getattr(rec, "text", "") or ""
            text = clean_text(raw)
            if not text:
                continue
            collected.append({
                "uri"        : post.uri,
                "author"     : post.author.handle,
                "display"    : getattr(post.author, "display_name", "") or "",
                "text"       : text,
                "timestamp"  : getattr(rec, "created_at", "") or "",
                "likes"      : post.like_count   or 0,
                "reposts"    : post.repost_count or 0,
                "replies"    : post.reply_count  or 0,
                "mentions"   : re.findall(r"@([\w.\-]+)", raw),
                "is_reply"   : bool(getattr(rec, "reply", None)),
                "post_type"  : "post",
                "parent_uri" : None,
                "query"      : query,
            })
        cursor = getattr(resp, "cursor", None)
        if not cursor: break
        time.sleep(0.5)
    return collected


# ── Comment fetcher ────────────────────────────────────────────────────────────
def fetch_comments(client, uri, depth=3):
    """
    Fetch all comments on a post recursively up to `depth` levels.
    Each comment stores the URI of its direct parent post/comment.
    """
    try:
        resp = client.app.bsky.feed.get_post_thread({"uri": uri, "depth": depth})
    except Exception as e:
        print(f"  Thread error: {e}")
        return []

    comments = []

    def walk(node, parent_uri=None):
        if node is None:
            return
        post = getattr(node, "post", None)
        if post is None:
            return
        rec         = getattr(post, "record", None)
        raw         = getattr(rec, "text", "") or ""
        text        = clean_text(raw)
        current_uri = post.uri
        if text and not is_news_bot(post.author.handle):
            comments.append({
                "uri"        : post.uri,
                "author"     : post.author.handle,
                "display"    : getattr(post.author, "display_name", "") or "",
                "text"       : text,
                "timestamp"  : getattr(rec, "created_at", "") or "",
                "likes"      : post.like_count   or 0,
                "reposts"    : post.repost_count or 0,
                "replies"    : post.reply_count  or 0,
                "mentions"   : re.findall(r"@([\w.\-]+)", raw),
                "is_reply"   : True,
                "post_type"  : "comment",
                "parent_uri" : parent_uri,
                "query"      : "thread",
            })
        for child in (getattr(node, "replies", None) or []):
            walk(child, parent_uri=current_uri)

    walk(getattr(resp, "thread", None))
    return comments


print("Helper functions ready: clean_text, is_news_bot, has_real_text, label_candidate, search_posts, fetch_comments")

In [18]:
# ── Scraping: alle hashtags + replies, geen limiet ────────────────────────────
RAW_CSV     = f"./Data/Bluesky/bsky_{ACTIVE_ELECTION}_raw.csv"
CLEANED_CSV = f"./Data/Bluesky/bsky_{ACTIVE_ELECTION}_posts.csv"

os.makedirs(os.path.dirname(os.path.abspath(RAW_CSV)), exist_ok=True)

all_posts = []
hashtags  = cfg["hashtags"]

# Stap 1: posts scrapen per hashtag
for i, tag in enumerate(hashtags, 1):
    print(f"[{i}/{len(hashtags)}] {tag} ...", end=" ", flush=True)
    posts = search_posts(client, tag, since=cfg["since"], until=cfg["until"])
    for p in posts:
        p["election"] = ACTIVE_ELECTION
    all_posts.extend(posts)
    print(f"{len(posts)} posts  (totaal: {len(all_posts):,})")

# Dedupliceren voor replies ophalen
df_posts = pd.DataFrame(all_posts).drop_duplicates(subset="uri").reset_index(drop=True)
print(f"\n{len(df_posts):,} unieke posts — nu replies ophalen...\n")

# Stap 2: replies ophalen per post
all_replies = []
for i, row in enumerate(df_posts.itertuples(), 1):
    if i % 100 == 0:
        print(f"  Replies: {i}/{len(df_posts)} posts verwerkt  ({len(all_replies):,} replies tot nu toe)")
    replies = fetch_replies(client, row.uri, depth=3)
    for r in replies:
        r["election"] = ACTIVE_ELECTION
    all_replies.extend(replies)

print(f"\nReplies opgehaald: {len(all_replies):,}")

# Stap 3: samenvoegen en dedupliceren
df_raw = pd.concat([df_posts, pd.DataFrame(all_replies)], ignore_index=True)
df_raw = df_raw.drop_duplicates(subset="uri").reset_index(drop=True)
df_raw.to_csv(RAW_CSV, index=False)

print(f"Raw opgeslagen: {len(df_raw):,} unieke posts+replies → {RAW_CSV}")

[1/62] #Election2024 ...   API error: 
0 posts  (totaal: 0)
[2/62] #USElection2024 ... 475 posts  (totaal: 475)
[3/62] #ElectionDay ... 220 posts  (totaal: 695)
[4/62] #Vote2024 ...   API error: 
0 posts  (totaal: 695)
[5/62] #Presidential2024 ... 1 posts  (totaal: 696)
[6/62] #PresidentialElection ... 135 posts  (totaal: 831)
[7/62] #ElectionNight ... 12 posts  (totaal: 843)
[8/62] #AmericaDecides ... 4 posts  (totaal: 847)
[9/62] #Decision2024 ... 62 posts  (totaal: 909)
[10/62] #VoterRegistration ... 80 posts  (totaal: 989)
[11/62] #EarlyVoting ...   API error: 
193 posts  (totaal: 1,182)
[12/62] #MailInVoting ... 7 posts  (totaal: 1,189)
[13/62] #ElectionIntegrity ... 19 posts  (totaal: 1,208)
[14/62] #Battleground2024 ... 0 posts  (totaal: 1,208)
[15/62] #SwingState ... 39 posts  (totaal: 1,247)
[16/62] #Trump2024 ...   API error: 
167 posts  (totaal: 1,414)
[17/62] #TrumpVance ... 73 posts  (totaal: 1,487)
[18/62] #VoteTrump ... 5 posts  (totaal: 1,492)
[19/62] #Trump ... 6655 po

In [20]:
# ── Comments opnieuw ophalen vanuit bestaande raw CSV (met parent_uri) ────────
# Gebruik deze cel als de raw CSV al bestaat maar parent_uri ontbreekt.
# Posts worden NIET opnieuw gescraped — alleen de comments worden opnieuw opgehaald.

if not os.path.exists(RAW_CSV):
    print("Geen raw CSV gevonden — voer eerst de scraping cel uit.")
else:
    df_existing = pd.read_csv(RAW_CSV)

    # Enkel de originele posts gebruiken als startpunt voor comments
    df_original_posts = df_existing[df_existing["post_type"] == "post"].drop_duplicates(subset="uri").reset_index(drop=True)
    print(f"{len(df_original_posts):,} posts gevonden in raw CSV — comments opnieuw ophalen...\n")

    # parent_uri toevoegen aan posts (is None voor originele posts)
    if "parent_uri" not in df_original_posts.columns:
        df_original_posts["parent_uri"] = None

    # Comments opnieuw ophalen met parent_uri
    all_replies = []
    for i, row in enumerate(df_original_posts.itertuples(), 1):
        if i % 100 == 0:
            print(f"  {i}/{len(df_original_posts)} posts verwerkt  ({len(all_replies):,} comments tot nu toe)")
        replies = fetch_replies(client, row.uri, depth=3)
        for r in replies:
            r["election"] = ACTIVE_ELECTION
        all_replies.extend(replies)

    print(f"\nComments opgehaald: {len(all_replies):,}")

    # Samenvoegen: originele posts + nieuwe comments (met parent_uri)
    df_raw = pd.concat([df_original_posts, pd.DataFrame(all_replies)], ignore_index=True)
    df_raw = df_raw.drop_duplicates(subset="uri").reset_index(drop=True)
    df_raw.to_csv(RAW_CSV, index=False)

    print(f"Raw CSV bijgewerkt: {len(df_raw):,} unieke rijen → {RAW_CSV}")
    print(f"Kolommen: {list(df_raw.columns)}")

24,811 posts gevonden in raw CSV — comments opnieuw ophalen...

  100/24811 posts verwerkt  (135 comments tot nu toe)
  200/24811 posts verwerkt  (244 comments tot nu toe)
  300/24811 posts verwerkt  (383 comments tot nu toe)
  400/24811 posts verwerkt  (506 comments tot nu toe)
  500/24811 posts verwerkt  (647 comments tot nu toe)
  600/24811 posts verwerkt  (825 comments tot nu toe)
  700/24811 posts verwerkt  (979 comments tot nu toe)
  800/24811 posts verwerkt  (1,113 comments tot nu toe)
  900/24811 posts verwerkt  (1,276 comments tot nu toe)
  1000/24811 posts verwerkt  (1,397 comments tot nu toe)
  1100/24811 posts verwerkt  (1,533 comments tot nu toe)
  1200/24811 posts verwerkt  (1,662 comments tot nu toe)
  1300/24811 posts verwerkt  (1,804 comments tot nu toe)
  1400/24811 posts verwerkt  (1,936 comments tot nu toe)
  1500/24811 posts verwerkt  (2,069 comments tot nu toe)
  1600/24811 posts verwerkt  (2,214 comments tot nu toe)
  1700/24811 posts verwerkt  (2,359 comments to

In [21]:
# ── Cleaning: filter, labeling en opslaan ────────────────────────────────────
if not os.path.exists(RAW_CSV):
    print("Geen raw CSV gevonden — voer eerst de scraping cel uit.")
else:
    df = pd.read_csv(RAW_CSV)
    before = len(df)

    # Timestamps parsen
    df["datetime"] = pd.to_datetime(df["timestamp"], format="ISO8601", utc=True)

    # Filter binnen het verkiezingsvenster
    df = df[
        (df["datetime"] >= cfg["since"]) &
        (df["datetime"] <= cfg["until"])
    ]

    # Filter nieuwsbots en te korte teksten
    df = df[~df["author"].apply(is_news_bot)]
    df = df[df["text"].apply(has_real_text)]

    # Kandidaat labelen
    df["candidate"] = df.apply(label_candidate, axis=1)

    # Chronologisch sorteren
    df = df.sort_values("datetime").reset_index(drop=True)

    # Kolomvolgorde (parent_uri toegevoegd)
    cols = ["timestamp", "author", "display", "text", "candidate", "post_type",
            "likes", "reposts", "replies", "mentions", "is_reply", "parent_uri",
            "query", "uri", "election"]
    cols = [c for c in cols if c in df.columns]
    df = df[cols]

    df.to_csv(CLEANED_CSV, index=False)

    print(f"Gefilterd : {before:,} → {len(df):,} posts  ({before - len(df):,} verwijderd)")
    print(f"Datumbereik: {df.timestamp.iloc[0][:10]} → {df.timestamp.iloc[-1][:10]}")
    print(f"Kandidaten : {df.candidate.value_counts().to_dict()}")
    print(f"Cleaned opgeslagen → {CLEANED_CSV}")

Gefilterd : 32,862 → 27,215 posts  (5,647 verwijderd)
Datumbereik: 2024-07-05 → 2024-11-04
Kandidaten : {'CandidateA': 11749, 'Neutral': 9532, 'CandidateB': 5934}
Cleaned opgeslagen → ./Data/Bluesky/bsky_US_2024_posts.csv


In [12]:
# label_candidate is defined in cell t07 above
print("label_candidate ready ✓")


label_candidate ready ✓
